# EDA - cross-table features (`features/cross.py`)

Validates the 8 cross-table feature recommendations from the EDA series.
The features themselves are implemented in `features/cross.py`; this
notebook re-loads them, profiles their distributions, evaluates their
bivariate signal vs `scored_after`, tests redundancy and significance,
and finalises a keep / drop manifest.

| # | Feature | Source mechanism |
|---|---------|------------------|
| 1 | `shots_per_top_third_run`              | clinical-finisher proxy |
| 2 | `top_third_run_share_{A,M,D,G}`        | position x spatial role |
| 3 | `last15_sprints_{A,M,D,G}`             | recovers attacker-fatigue inversion |
| 4 | `fatigue_gap`                          | cumul - last15 sprint intensity |
| 5 | `top_third_run_share_x_set_piece_share`| target-striker profile |
| 6 | `top_third_run_share_x_shot_accuracy`  | clinical attacking-third presence |
| 7 | `shot_intent_under_fatigue`            | tired but still shooting |
| 8 | `speed_above_team_avg`                 | relative speed (team-in-match) |

Section structure:

| ID | Section |
|----|---------|
| A | Build & profile (univariate description) |
| B | Bivariate signal vs `scored_after` (pooled + by position) |
| C | Redundancy check (correlation among the 8) |
| D | Cluster-robust GLM + BH-FDR significance |
| E | Final manifest |


## Setup

In [1]:
"""Imports."""
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Allow running from `eda/` directory.
PROJECT_ROOT: Path = Path.cwd().parent if Path.cwd().name == "eda" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from features.cross import build_cross_features, CROSS_FEATURE_COLUMNS


In [2]:
"""Display options."""
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 80)
pd.set_option("display.precision", 4)


In [3]:
"""Load source CSVs."""
DATA_DIR = PROJECT_ROOT / "data"
main  = pd.read_csv(DATA_DIR / "players_quarters_final.csv")
runs  = pd.read_csv(DATA_DIR / "player_appearance_run.csv")
shots = pd.read_csv(DATA_DIR / "player_appearance_shot_limited.csv")
main.shape, runs.shape, shots.shape


((3486, 33), (35133, 10), (780, 12))

In [4]:
"""Helpers (Wilson CI + binned-rate table) - identical to the run EDA."""
from scipy import stats as _stats


def wilson_ci(successes: int, trials: int, alpha: float = 0.05) -> tuple[float, float]:
    if trials == 0:
        return float("nan"), float("nan")
    p = successes / trials
    z = _stats.norm.ppf(1.0 - alpha / 2.0)
    denom = 1.0 + (z ** 2) / trials
    centre = (p + (z ** 2) / (2.0 * trials)) / denom
    half = z * np.sqrt(p * (1.0 - p) / trials + (z ** 2) / (4.0 * trials ** 2)) / denom
    return float(centre - half), float(centre + half)


def binned_rate(df: pd.DataFrame, feature: str, target: str = "scored_after",
                bins=None, by: str | None = None) -> pd.DataFrame:
    work = df[[feature, target] + ([by] if by else [])].copy()
    if bins is not None:
        work["__bin"] = pd.cut(work[feature], bins=bins, include_lowest=True)
    else:
        work["__bin"] = work[feature]
    grouping = ["__bin"] if by is None else [by, "__bin"]
    g = work.groupby(grouping, observed=False, dropna=False)[target].agg(["size", "sum"])
    g = g.rename(columns={"size": "n", "sum": "n_pos"}).reset_index()
    g["rate"] = g["n_pos"] / g["n"].where(g["n"] > 0)
    ci = g.apply(lambda r: wilson_ci(int(r["n_pos"]), int(r["n"])), axis=1, result_type="expand")
    g["ci_lo"], g["ci_hi"] = ci[0], ci[1]
    return g.rename(columns={"__bin": "bin"})


## Section A - Build & profile

We invoke the productionised builder and verify (a) the output shape matches
the main panel and (b) every feature has a sensible univariate distribution.


In [5]:
"""Build the cross feature frame."""
cross = build_cross_features(main, runs, shots)
print(f"shape: {cross.shape}")
assert cross.shape[0] == len(main), "row count must match the main panel"
assert list(cross.columns) == list(CROSS_FEATURE_COLUMNS)
cross.head(3)


shape: (3486, 16)


,player_appearance_id,checkpoint,shots_per_top_third_run,top_third_run_share_A,top_third_run_share_M,top_third_run_share_D,top_third_run_share_G,last15_sprints_A,last15_sprints_M,last15_sprints_D,last15_sprints_G,fatigue_gap,top_third_run_share_x_set_piece_share,top_third_run_share_x_shot_accuracy,shot_intent_under_fatigue,speed_above_team_avg
0,39421,H1_15,NaN,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,-6.1565
1,39422,H1_15,NaN,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,NaN,NaN,0.0,0.6495
2,39423,H1_15,NaN,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,NaN,NaN,0.0,1.0085


In [6]:
"""Univariate description (numeric features only)."""
NUMERIC = [c for c in cross.columns if c not in ("player_appearance_id", "checkpoint")]
desc = cross[NUMERIC].describe().T[["count", "mean", "std", "min", "50%", "max"]]
desc["n_nan"] = len(cross) - desc["count"].astype(int)
desc.round(4)


,count,mean,std,min,50%,max,n_nan
shots_per_top_third_run,2928.0,0.0755,0.1703,0.0000,0.0000,2.0000,558
top_third_run_share_A,3486.0,0.0801,0.1891,0.0000,0.0000,1.0000,0
top_third_run_share_M,3484.0,0.1158,0.1805,0.0000,0.0000,1.0000,2
top_third_run_share_D,3484.0,0.0649,0.1197,0.0000,0.0000,1.0000,2
top_third_run_share_G,3367.0,0.0027,0.0467,0.0000,0.0000,1.0000,119
last15_sprints_A,3486.0,0.3655,1.0296,0.0000,0.0000,8.0000,0
last15_sprints_M,3486.0,0.6079,1.1488,0.0000,0.0000,7.0000,0
last15_sprints_D,3486.0,0.5112,1.0449,0.0000,0.0000,6.0000,0
last15_sprints_G,3486.0,0.0063,0.0792,0.0000,0.0000,1.0000,0
fatigue_gap,3486.0,0.0060,0.0623,-0.2667,0.0000,0.5500,0


### A - findings

Sanity-check expectations:

* `top_third_run_share_*` and `last15_sprints_*` columns are mostly zero
  (one column is non-zero per row, gated by position).
* `fatigue_gap` centred near zero, signed range covering both "ramping up"
  (<0) and "slowing down" (>0).
* `speed_above_team_avg` should average ~0 per fixture-side by construction;
  the global mean drifts only because the panel pools many fixtures with
  unequal coverage.
* NaN counts are non-zero on ratio features only - per the module's NaN
  policy (zero denominators preserved as NaN for downstream imputation).


## Section B - Bivariate signal vs `scored_after`

We merge the cross frame onto the main panel and compute (i) Pearson
correlation and (ii) binned target rate with Wilson 95% CI for each
feature. Strong, monotone, and robust signals get advanced; flat or
counter-directional ones get demoted.


In [7]:
"""Merge for analysis."""
m = main[["player_appearance_id", "checkpoint", "scored_after", "fixture_id", "position"]].merge(
    cross, on=["player_appearance_id", "checkpoint"], how="left"
)
BASELINE = m["scored_after"].mean()
print(f"baseline scored_after rate: {BASELINE:.4f}")


baseline scored_after rate: 0.0582


In [8]:
"""Pearson r vs target (NaN-pairwise drop)."""
rows = []
for c in NUMERIC:
    s = m[c].dropna()
    if s.std() == 0:
        rows.append({"feature": c, "n": len(s), "pearson_r": float("nan")})
        continue
    t = m.loc[s.index, "scored_after"]
    rows.append({"feature": c, "n": len(s), "pearson_r": float(np.corrcoef(s, t)[0, 1])})
pearson_df = pd.DataFrame(rows).sort_values("pearson_r", key=lambda x: x.abs(), ascending=False)
pearson_df.reset_index(drop=True)


,feature,n,pearson_r
0,top_third_run_share_A,3486,0.1291
1,top_third_run_share_D,3484,-0.0741
2,last15_sprints_D,3486,-0.0724
3,last15_sprints_A,3486,0.0712
4,fatigue_gap,3486,0.0611
5,top_third_run_share_x_set_piece_share,1014,0.0601
6,shot_intent_under_fatigue,3486,0.0546
7,speed_above_team_avg,3486,0.0472
8,top_third_run_share_M,3484,0.0408
9,shots_per_top_third_run,2928,0.0383


In [9]:
"""Binned target rates with Wilson CIs (chosen per feature shape)."""
BINS = {
    "shots_per_top_third_run":              [-1, 0, 0.05, 0.2, 3],
    "top_third_run_share_A":                [-1, 0, 0.1, 0.3, 1.01],
    "top_third_run_share_M":                [-1, 0, 0.1, 0.3, 1.01],
    "top_third_run_share_D":                [-1, 0, 0.1, 0.3, 1.01],
    "top_third_run_share_G":                [-1, 0, 0.1, 1.01],
    "last15_sprints_A":                     [-1, 0, 1, 3, 100],
    "last15_sprints_M":                     [-1, 0, 1, 3, 100],
    "last15_sprints_D":                     [-1, 0, 1, 3, 100],
    "last15_sprints_G":                     [-1, 0, 100],
    "fatigue_gap":                          [-1, -0.05, 0.05, 1],
    "top_third_run_share_x_set_piece_share":[-1, 0, 0.1, 0.3, 1.01],
    "top_third_run_share_x_shot_accuracy":  [-1, 0, 0.1, 0.3, 1.01],
    "shot_intent_under_fatigue":            [-1, -0.01, 0.01, 1],
    "speed_above_team_avg":                 [-10, -0.3, 0.3, 10],
}

frames = []
for feat, edges in BINS.items():
    t = binned_rate(m, feat, "scored_after", bins=edges)
    t["feature"] = feat
    frames.append(t)
binned_df = pd.concat(frames, ignore_index=True)[["feature", "bin", "n", "n_pos", "rate", "ci_lo", "ci_hi"]]
binned_df


,feature,bin,n,n_pos,rate,ci_lo,ci_hi
0,shots_per_top_third_run,"(-1.001, 0.0]",1926,110,0.0571,4.7604e-02,0.0684
1,shots_per_top_third_run,"(0.0, 0.05]",80,3,0.0375,1.2835e-02,0.1045
2,shots_per_top_third_run,"(0.05, 0.2]",604,44,0.0728,5.4711e-02,0.0964
3,shots_per_top_third_run,"(0.2, 3.0]",318,30,0.0943,6.6882e-02,0.1315
4,shots_per_top_third_run,NaN,558,16,0.0287,1.7726e-02,0.0461
5,top_third_run_share_A,"(-1.001, 0.0]",2880,128,0.0444,3.7505e-02,0.0526
6,top_third_run_share_A,"(0.0, 0.1]",2,1,0.5000,9.4531e-02,0.9055
7,top_third_run_share_A,"(0.1, 0.3]",105,16,0.1524,9.6027e-02,0.2333
8,top_third_run_share_A,"(0.3, 1.01]",499,58,0.1162,9.0998e-02,0.1473
9,top_third_run_share_M,"(-1.001, 0.0]",2225,118,0.0530,4.4469e-02,0.0631


### B - findings

Read off the table above. Expected patterns from the prior reasoning:

* `top_third_run_share_A` should show the steepest monotone climb (it was
  the strongest feature in the cross set).
* `fatigue_gap` should show a clean three-bucket pattern: lowest rate in the
  "slowing down" bucket, highest in the "ramping up" bucket.
* `top_third_run_share_D` should be **negative** (defenders pushing high =
  tactical anomaly, not a goal-scoring positioner).
* `speed_above_team_avg` should be monotone from "slow" to "fast".

Anything flat across all bins becomes a drop candidate even if its
linear-r looks borderline.


## Section C - Redundancy among the 8 features

By construction several of these features share parents (e.g.
`top_third_run_share_*` shares the underlying ratio with the multiplicative
interactions in #5 / #6). Pearson correlations above 0.9 mean we should
keep only one of the pair for linear models; tree models can tolerate the
collinearity natively.


In [10]:
"""Pearson correlation matrix on the numeric cross features."""
corr = cross[NUMERIC].corr(method="pearson")
hi = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .rename("pearson")
        .reset_index()
        .rename(columns={"level_0": "a", "level_1": "b"})
)
hi["abs"] = hi["pearson"].abs()
hi.sort_values("abs", ascending=False).head(15).reset_index(drop=True)


,a,b,pearson,abs
0,top_third_run_share_A,last15_sprints_A,0.6884,0.6884
1,top_third_run_share_D,last15_sprints_D,0.6035,0.6035
2,top_third_run_share_M,last15_sprints_M,0.6013,0.6013
3,fatigue_gap,shot_intent_under_fatigue,0.4004,0.4004
4,top_third_run_share_M,top_third_run_share_D,-0.3482,0.3482
5,top_third_run_share_M,last15_sprints_D,-0.3140,0.3140
6,top_third_run_share_D,last15_sprints_M,-0.2871,0.2871
7,last15_sprints_D,fatigue_gap,-0.2825,0.2825
8,top_third_run_share_A,top_third_run_share_M,-0.2720,0.2720
9,last15_sprints_M,last15_sprints_D,-0.2589,0.2589


### C - findings

Expected near-collinearity:

* `top_third_run_share_x_set_piece_share` and
  `top_third_run_share_x_shot_accuracy` will both share most of their
  variance with their `top_third_run_share_*` parent in any subgroup
  where the second factor is roughly constant.
* `top_third_run_share_A` and `last15_sprints_A` move together for
  attackers (both gated on the same dummy).
* `shot_intent_under_fatigue` shares variance with `fatigue_gap`.

Decision logic: keep the feature with the higher single-bivariate signal
and drop the dependent partner unless a tree model's importance ranking
later contradicts it.


## Section D - Cluster-robust significance + BH-FDR

We fit a one-feature logistic regression of `scored_after` on each cross
feature with cluster-robust standard errors (cluster on `fixture_id`) and
adjust the resulting p-values via Benjamini-Hochberg FDR across the 14
columns. Survives at q=0.05 = "this feature is non-zero even after
honouring the multiple-players-per-match clustering and the multiple-test
burden".


In [11]:
"""Cluster-robust GLM + BH-FDR."""
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests


def cluster_robust_p(df: pd.DataFrame, feature: str) -> tuple[float, float, int]:
    sub = df[[feature, "scored_after", "fixture_id"]].dropna()
    if sub[feature].std() == 0 or len(sub) < 30:
        return float("nan"), float("nan"), len(sub)
    try:
        fitted = smf.logit(f"scored_after ~ {feature}", data=sub).fit(
            disp=False, maxiter=200,
            cov_type="cluster", cov_kwds={"groups": sub["fixture_id"]},
        )
        return float(fitted.params[feature]), float(fitted.pvalues[feature]), len(sub)
    except Exception:
        return float("nan"), float("nan"), len(sub)


rows = []
for f in NUMERIC:
    coef, p, n = cluster_robust_p(m, f)
    r = m[[f, "scored_after"]].corr().iloc[0, 1]
    rows.append({"feature": f, "n": n, "pearson_r": r, "coef": coef, "p_raw": p})
sig = pd.DataFrame(rows)

mask = sig["p_raw"].notna()
adj = np.full(len(sig), np.nan)
if mask.sum():
    _, adj_p, _, _ = multipletests(sig.loc[mask, "p_raw"].values, method="fdr_bh")
    adj[mask.values] = adj_p
sig["p_bh"] = adj
sig["bh_q05"] = sig["p_bh"] < 0.05
sig.sort_values("p_bh").reset_index(drop=True)


C:\Users\tymot\projects\wec\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\tymot\projects\wec\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,feature,n,pearson_r,coef,p_raw,p_bh,bh_q05
0,last15_sprints_G,3486,-0.0198,-22.3406,0.0000e+00,0.0000e+00,True
1,top_third_run_share_G,3367,-0.0148,-117.1185,5.8133e-81,4.0693e-80,True
2,top_third_run_share_A,3486,0.1291,2.0363,5.8166e-09,2.7144e-08,True
3,fatigue_gap,3486,0.0611,3.6382,2.4480e-04,8.5679e-04,True
4,last15_sprints_A,3486,0.0712,0.2141,7.1316e-04,1.9969e-03,True
5,shot_intent_under_fatigue,3486,0.0546,5.6193,9.0041e-04,2.1010e-03,True
6,speed_above_team_avg,3486,0.0472,0.2631,5.9243e-03,1.1849e-02,True
7,last15_sprints_D,3486,-0.0724,-0.4852,3.1002e-02,5.4254e-02,False
8,last15_sprints_M,3486,0.0380,0.1239,5.5219e-02,7.7306e-02,False
9,top_third_run_share_D,3484,-0.0741,-3.9713,4.9872e-02,7.7306e-02,False


### D - findings

Three classes of feature:

1. **Survives BH q=0.05** -> goes into the final manifest unconditionally.
2. **Bivariate-significant on Pearson but fails BH** -> within-fixture
   clustering eats the apparent signal. Demote unless tree-model
   importance later rescues it.
3. **Pearson borderline + non-significant BH** -> drop.

The most likely BH survivors based on prior bivariate work are
`top_third_run_share_A`, `fatigue_gap`, and `speed_above_team_avg`. The
position-zero columns (`*_G`, `last15_sprints_G`) will fail trivially -
goalkeepers contribute deterministic zeros to the target.


## Section E - Final manifest

Combines:

* the binned-rate verdict from B (shape of the relationship),
* the redundancy verdict from C (which pair-sibling to drop),
* the cluster-robust significance from D.

The output is a DataFrame keyed by feature with the keep / drop decision
and the headline numbers that justify it.


In [12]:
"""Programmatic manifest."""
def best_non_baseline_bin(df: pd.DataFrame, feature: str) -> tuple[float, float, float, int]:
    sub = df[df["feature"] == feature]
    if sub.empty:
        return (float("nan"),) * 3 + (0,)
    sub = sub.sort_values("rate", ascending=False).iloc[0]
    return float(sub["rate"]), float(sub["ci_lo"]), float(sub["ci_hi"]), int(sub["n"])


rows = []
for feat in NUMERIC:
    rate, lo, hi, n = best_non_baseline_bin(binned_df, feat)
    rows.append({"feature": feat, "best_rate": rate, "ci_lo": lo, "ci_hi": hi, "n": n})

manifest = pd.DataFrame(rows).merge(
    sig[["feature", "pearson_r", "p_bh", "bh_q05"]], on="feature", how="left"
)


def verdict(row) -> str:
    """Final verdict.

    Robust against statsmodels' spurious p_bh = 0 on perfectly-separating
    columns (e.g. goalkeepers, where the target is identically zero).
    Requires the strongest bin rate to materially exceed the baseline before
    accepting BH significance.
    """
    if pd.isna(row["best_rate"]):
        return "drop (no data)"
    rate_above_baseline = row["best_rate"] > BASELINE * 1.2
    ci_above_baseline = (
        not pd.isna(row["ci_lo"]) and row["ci_lo"] > BASELINE
    )
    if row["bh_q05"] is True and (rate_above_baseline or ci_above_baseline):
        return "KEEP"
    if ci_above_baseline:
        return "keep (cautious)"
    if row["best_rate"] > BASELINE * 1.4 and row["n"] >= 30:
        return "keep (cautious)"
    return "drop (flat)"


manifest["verdict"] = manifest.apply(verdict, axis=1)
manifest.sort_values(["verdict", "p_bh"]).reset_index(drop=True)


,feature,best_rate,ci_lo,ci_hi,n,pearson_r,p_bh,bh_q05,verdict
0,top_third_run_share_A,0.5000,0.0945,0.9055,2,0.1291,2.7144e-08,True,KEEP
1,fatigue_gap,0.0711,0.0532,0.0944,605,0.0611,8.5679e-04,True,KEEP
2,last15_sprints_A,0.1500,0.1029,0.2135,160,0.0712,1.9969e-03,True,KEEP
3,shot_intent_under_fatigue,0.1106,0.0748,0.1605,208,0.0546,2.1010e-03,True,KEEP
4,speed_above_team_avg,0.0700,0.0575,0.0850,1328,0.0472,1.1849e-02,True,KEEP
5,last15_sprints_G,0.0586,0.0513,0.0669,3464,-0.0198,0.0000e+00,True,drop (flat)
6,top_third_run_share_G,0.0605,0.0530,0.0691,3353,-0.0148,4.0693e-80,True,drop (flat)
7,top_third_run_share_M,0.0727,0.0549,0.0956,633,0.0408,1.7193e-01,False,drop (flat)
8,last15_sprints_D,0.0681,0.0590,0.0785,2584,-0.0724,5.4254e-02,False,keep (cautious)
9,top_third_run_share_D,0.0723,0.0627,0.0833,2435,-0.0741,7.7306e-02,False,keep (cautious)


### Closing notes

* The DataFrame above is the deliverable of this notebook. Each cross
  feature has been re-validated against the data after the engineering
  step, so the keep / drop verdict is consistent with the actual values
  produced by `features/cross.py`.
* The 10% replication gap on `last15_*` runs (run-EDA Section B) propagates
  into `fatigue_gap` and `shot_intent_under_fatigue` indirectly via
  `last15_sprints` from main. If those two features survive the manifest,
  reconcile the windowing rule before promoting them past pilot models.
* For modelling: pass the cross features as a separate feature group so
  ablation experiments can quantify their incremental contribution over
  the shots + runs baseline (relevant to RQ3 / RQ4).
